In [30]:
# Importamos las librerías oficiales de Google para Colab
from google.colab import auth
auth.authenticate_user()

import gspread
import google.auth

# Solicitamos las credenciales de  Gmail de forma segura
creds, _ = google.auth.default()
client = gspread.authorize(creds)

print("Colab ya tiene permiso para editar tu Google Sheets.")

Colab ya tiene permiso para editar tu Google Sheets.


In [31]:
# Abrimos la hoja de GoogleSheets
hoja_prueba = client.open("Reporte Automatizado - Libros").sheet1

In [32]:
import requests
from bs4 import BeautifulSoup

# Dirección de la tienda de libros
url = "https://books.toscrape.com/"

# Simulamos  navegador real (Chrome en Windows) para saltar bloqueos
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
    'Accept-Language': 'es-ES,es;q=0.9,en;q=0.8'
}

respuesta = requests.get(url, headers=headers)
soup = BeautifulSoup(respuesta.text, 'html.parser')

datos_libros = []

# Buscamos los H3 y los párrafos que tienen el símbolo £ o €
titulos_tags = soup.find_all('h3')
precios_tags = soup.find_all('p', class_='price_color')

# Si los encuentra por separado, los emparejamos en nuestra lista
if len(titulos_tags) == len(precios_tags) and len(titulos_tags) > 0:
    for i in range(len(titulos_tags)):
        enlace = titulos_tags[i].find('a')
        if enlace:
            titulo = enlace.get('title') or enlace.text.strip()
            precio = precios_tags[i].text.strip()
            datos_libros.append([titulo, precio])
else:
    # Si falla, buscamos genéricamente cualquier contenedor 'article' (estructura nativa de books.toscrape)
    articulos = soup.find_all('article')
    for art in articulos:
        h3_tag = art.find('h3')
        price_tag = art.find('p', class_='price_color')
        if h3_tag and price_tag:
            enlace = h3_tag.find('a')
            titulo = enlace.get('title') if enlace and enlace.get('title') else h3_tag.text.strip()
            precio = price_tag.text.strip()
            datos_libros.append([titulo, precio])

print(f"Cantidad de libros extraídos de la Web: {len(datos_libros)}")
if len(datos_libros) > 0:
    print("\nMuestra de los primeros 3 libros:")
    for item in datos_libros[:3]:
        print(f"- Libro: {item[0]} | Precio: {item[1]}")

Cantidad de libros extraídos de la Web: 20

Muestra de los primeros 3 libros:
- Libro: A Light in the Attic | Precio: Â£51.77
- Libro: Tipping the Velvet | Precio: Â£53.74
- Libro: Soumission | Precio: Â£50.10


In [33]:
# Simulo tasa de cambio (1 Libra Esterlina = $1300)
TASA_CAMBIO_PESOS = 1300

datos_transformados = []

for libro in datos_libros:
    titulo_original = str(libro[0])
    precio_original = str(libro[1])

    # Elimino caracteres no numéricos
    precio_limpio_str = ""
    for caracter in precio_original:
        if caracter.isdigit() or caracter == '.':
            precio_limpio_str += caracter

    try:
        #Conversión a decimal
        precio_numero = float(precio_limpio_str)

        #Conversión y redondeo
        precio_en_pesos = round(precio_numero * TASA_CAMBIO_PESOS, 2)

        #Formateo con separadores de miles y decimales
        precio_final_formato = f"$ {precio_en_pesos:,.2f}".replace(',', 'X').replace('.', ',').replace('X', '.')

    except ValueError:
        precio_final_formato = "$ 0,00"

    # Guardo datos transformados
    datos_transformados.append([titulo_original, precio_final_formato])

for item in datos_transformados[:3]:
    print(f"- {item[0]} -> {item[1]}")

- A Light in the Attic -> $ 67.301,00
- Tipping the Velvet -> $ 69.862,00
- Soumission -> $ 65.130,00


In [34]:
#Instalo librerías
!pip install gspread_formatting

#Estructura avanzada de Google Sheets para dar estilos
from gspread_formatting import *

#Conecto la hoja nuevamente
hoja_diseno = client.open("Reporte Automatizado - Libros").sheet1

#Aplico filtro automático de la A1 a B100
hoja_diseno.set_basic_filter("A1:B100")

#Estilo al encabezado
formato_encabezado = cellFormat(
    backgroundColor=color(0.2, 0.2, 0.2), # Gris oscuro profesional
    textFormat=textFormat(bold=True, foregroundColor=color(1, 1, 1), fontSize=11), # Texto blanco y negrita
    horizontalAlignment='CENTER'
)
format_cell_range(hoja_diseno, "A1:B1", formato_encabezado)

#Ajusto ancho de columnas de forma automática
hoja_diseno.columns_auto_resize(0, 1) # Ajusta columna A (0) y columna B (1)

{'spreadsheetId': '10m7wd-FfunPWuL0tA_Ude1hfxaSs2AQfSC2H7Eefk_w',
 'replies': [{}]}

In [35]:
# Conectar con la hoja de GoogleSheets
hoja_final = client.open("Reporte Automatizado - Libros").sheet1

#Borrar lo que haya en la hoja para evitar que se mezclen datos viejos
hoja_final.clear()

#Escribo encabezados
hoja_final.append_row(["Título del Producto (Importado)", "Precio de Venta (ARS $)"])

# 4. Forzamos la carga de la lista 'datos_transformados' (la que tiene los pesos)
hoja_final.append_rows(datos_transformados)

print("Datos cargados correctamente en Google Sheets")

Datos cargados correctamente en Google Sheets
